In [1]:
import os
import sys
import random
import copy
import datetime

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

import cv2


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(42)

DATA_ROOT = "/kaggle/input/cassava-leaf-disease-classification"
TRAIN_DIR = os.path.join(DATA_ROOT, "train_images")
TEST_DIR = os.path.join(DATA_ROOT, "test_images")
SAMPLE_SUB_PATH = os.path.join(DATA_ROOT, "sample_submission.csv")
TRAIN_CSV_PATH = os.path.join(DATA_ROOT, "train.csv")

print("Found train dir:", os.path.isdir(TRAIN_DIR), TRAIN_DIR)
print("Found test dir:", os.path.isdir(TEST_DIR), TEST_DIR)
print("Found sample_submission:", os.path.isfile(SAMPLE_SUB_PATH), SAMPLE_SUB_PATH)
print("Found train.csv:", os.path.isfile(TRAIN_CSV_PATH), TRAIN_CSV_PATH)

sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
assert list(sample_sub.columns) == ["image_id", "label"]
print("sample_submission rows:", len(sample_sub))

train_df = pd.read_csv(TRAIN_CSV_PATH)
assert list(train_df.columns) == ["image_id", "label"]
print("train rows:", len(train_df))
print(train_df["label"].value_counts().sort_index())



Found train dir: True /kaggle/input/cassava-leaf-disease-classification/train_images
Found test dir: True /kaggle/input/cassava-leaf-disease-classification/test_images
Found sample_submission: True /kaggle/input/cassava-leaf-disease-classification/sample_submission.csv
Found train.csv: True /kaggle/input/cassava-leaf-disease-classification/train.csv
sample_submission rows: 2676
train rows: 18721
label
0      939
1     1901
2     2091
3    11523
4     2267
Name: count, dtype: int64


In [2]:
import torchvision
import torchvision.transforms as T
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Keep the same core model (ResNet50) but actually train the 5-class head (and optionally last stage)
model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 5)
model = model.to(device)

# Minimal transforms: keep your validation/inference pipeline, add light augmentation for training
train_tfms = T.Compose(
    [
        T.ToPILImage(),
        T.Resize(512),
        T.RandomResizedCrop(448, scale=(0.85, 1.0)),
        T.RandomHorizontalFlip(p=0.5),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]
)

val_tfms = T.Compose(
    [
        T.ToPILImage(),
        T.Resize(512),
        T.CenterCrop(448),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]
)




Device: cuda


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:01<00:00, 64.7MB/s]


In [3]:
class CassavaTrainSet(Dataset):
    def __init__(self, df, img_dir, transforms=None):
        self.df = df.reset_index(drop=True).copy()
        self.img_dir = img_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fn = self.df.loc[idx, "image_id"]
        y = int(self.df.loc[idx, "label"])
        path = os.path.join(self.img_dir, fn)
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Failed to read image: {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transforms is not None:
            img = self.transforms(img)
        return img, y


class InferSet(Dataset):
    def __init__(self, image_ids, img_dir, transforms=None):
        self.image_ids = list(image_ids)
        self.img_dir = img_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        fn = self.image_ids[idx]
        path = os.path.join(self.img_dir, fn)
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Failed to read image: {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transforms is not None:
            img = self.transforms(img)
        return img, fn


# Minimal, deterministic split for quick sanity validation during training
from sklearn.model_selection import StratifiedShuffleSplit

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
tr_idx, va_idx = next(sss.split(train_df["image_id"].values, train_df["label"].values))
tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
va_df = train_df.iloc[va_idx].reset_index(drop=True)
print("train/val sizes:", len(tr_df), len(va_df))
print("val label dist:\n", va_df["label"].value_counts(normalize=True).sort_index())

train_ds = CassavaTrainSet(tr_df, TRAIN_DIR, transforms=train_tfms)
val_ds = CassavaTrainSet(va_df, TRAIN_DIR, transforms=val_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

x0, y0 = next(iter(train_loader))
print("Train batch:", x0.shape, y0[:8].tolist())



train/val sizes: 16848 1873
val label dist:
 label
0    0.050187
1    0.101442
2    0.111586
3    0.615590
4    0.121196
Name: proportion, dtype: float64
Train batch: torch.Size([32, 3, 448, 448]) [3, 1, 4, 1, 3, 0, 3, 3]


In [4]:
# Change directly aimed at improving accuracy: actually train on provided labeled data.
# Still standard cross-entropy; we add class weights to counter imbalance (same loss family/semantics).
import torch.nn.functional as F

# Freeze most layers to keep training fast and stable; fine-tune only last block + classifier
for p in model.parameters():
    p.requires_grad = False
for p in model.layer4.parameters():
    p.requires_grad = True
for p in model.fc.parameters():
    p.requires_grad = True

# Class weights from train distribution
counts = tr_df["label"].value_counts().sort_index()
counts = counts.reindex(range(5), fill_value=0).values.astype(np.float32)
weights = counts.sum() / np.clip(counts, 1.0, None)
weights = weights / weights.mean()
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
print("class_weights:", class_weights.detach().cpu().numpy())

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=3e-4,
    weight_decay=1e-4,
)


def accuracy_from_logits(logits, y):
    pred = torch.argmax(logits, dim=1)
    return (pred == y).float().mean().item()


epochs = (
    3  # small but meaningful improvement over random head; keeps within time budget
)

for epoch in range(1, epochs + 1):
    model.train()
    tr_loss = 0.0
    tr_acc = 0.0
    n_tr = 0

    for images, y in train_loader:
        images = images.to(device, non_blocking=True)
        y = torch.as_tensor(y, device=device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        bs = images.size(0)
        tr_loss += loss.item() * bs
        tr_acc += accuracy_from_logits(logits.detach(), y) * bs
        n_tr += bs

    model.eval()
    va_loss = 0.0
    va_acc = 0.0
    n_va = 0
    with torch.no_grad():
        for images, y in val_loader:
            images = images.to(device, non_blocking=True)
            y = torch.as_tensor(y, device=device)
            logits = model(images)
            loss = criterion(logits, y)

            bs = images.size(0)
            va_loss += loss.item() * bs
            va_acc += accuracy_from_logits(logits, y) * bs
            n_va += bs

    print(
        f"Epoch {epoch}/{epochs} | "
        f"train_loss={tr_loss/n_tr:.4f} train_acc={tr_acc/n_tr:.4f} | "
        f"val_loss={va_loss/n_va:.4f} val_acc={va_acc/n_va:.4f}"
    )



class_weights: [2.0503953  1.0126148  0.92060786 0.16707656 0.84930587]
Epoch 1/3 | train_loss=0.8029 train_acc=0.7777 | val_loss=0.6541 val_acc=0.8334
Epoch 2/3 | train_loss=0.5998 train_acc=0.8373 | val_loss=0.6485 val_acc=0.8243
Epoch 3/3 | train_loss=0.5146 train_acc=0.8517 | val_loss=0.6290 val_acc=0.8409


In [5]:
# Inference on the test set; keep your ordering/alignment guarantees against sample_submission.csv
test_image_ids = sample_sub["image_id"].tolist()
infer_ds = InferSet(test_image_ids, TEST_DIR, transforms=val_tfms)
infer_loader = DataLoader(
    infer_ds,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

all_fns = []
all_preds = []

model.eval()
with torch.no_grad():
    for images, fns in infer_loader:
        images = images.to(device, non_blocking=True)
        logits = model(images)
        preds = torch.argmax(logits, dim=1).detach().cpu().numpy().astype(int)

        all_fns.extend(list(fns))
        all_preds.extend(list(preds))

sub = pd.DataFrame({"image_id": all_fns, "label": all_preds})

sub = sample_sub[["image_id"]].merge(sub, on="image_id", how="left")
sub["label"] = sub["label"].fillna(0).astype(int)

assert len(sub) == len(sample_sub), (len(sub), len(sample_sub))
assert (
    sub["image_id"].tolist() == sample_sub["image_id"].tolist()
), "Order mismatch vs sample_submission"

out_path = "./submission.csv"
sub.to_csv(out_path, index=False)

print("Wrote:", out_path, "rows:", len(sub))
print(sub.head())



Wrote: ./submission.csv rows: 2676
         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      1
3  1234555380.jpg      1
4  1234571117.jpg      3


In [6]:
check = pd.read_csv("./submission.csv")
print(check.shape)
print(check.columns.tolist())
print(check["label"].value_counts().sort_index())
assert check.shape[0] == len(sample_sub)
assert check.columns.tolist() == ["image_id", "label"]
assert check["label"].between(0, 4).all()

(2676, 2)
['image_id', 'label']
label
0     189
1     343
2     361
3    1454
4     329
Name: count, dtype: int64
